# AND-105 Task 6: LLM Risk Explanation Prototype

**Model chosen:** `llama3.1:8b`  
**Why:** 8 B parameters balances local inference speed (~3–5 s/response on CPU) with output quality
for structured text generation. Smaller 3 B models produce less coherent multi-field reasoning;
70 B models exceed typical local GPU memory. `llama3.1:8b` demonstrates strong instruction-following
for constrained structured outputs, is natively supported by Ollama, and is available under a
permissive community licence.  

**Workflow:** Data gathering → V1 baseline prompt → `/branch` exploration (3 directions) →
side-by-side comparison → Writer/Reviewer session → final explanations for 10 elevators.

> **Note on DB connectivity:** On Windows with Docker Desktop, psycopg2 libpq 18 fails
> SCRAM-SHA-256 auth through the Docker NAT layer. Queries run via `docker exec psql`
> (Unix socket, trust) and results are returned as JSON. Functionally equivalent to
> a direct psycopg2 connection.

In [14]:
import asyncio, sys
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# Install required packages into the active kernel environment
!{sys.executable} -m pip install requests pandas --quiet


In [15]:
import json
import subprocess
from datetime import date, timedelta

import pandas as pd
import requests
from IPython.display import display, Markdown

CONTAINER  = 'rocket-elevators-ops-dashboard-db-1'
DB_USER    = 'api_user'
DB_NAME    = 'rocket_elevators'
OLLAMA_URL = 'http://localhost:11434'
MODEL      = 'llama3.1:8b'


def query_json(sql, params=None):
    '''Execute SQL inside the DB container via docker exec; return list of dicts.'''
    if params:
        for p in params:
            if isinstance(p, int):
                sql = sql.replace('%s', str(p), 1)
            else:
                safe = str(p).replace("'", "''")
                sql = sql.replace('%s', f"'{safe}'", 1)
    wrapped = f'SELECT json_agg(row_to_json(t)) FROM ({sql}) t'
    result = subprocess.run(
        ['docker', 'exec', CONTAINER,
         'psql', '-U', DB_USER, '-d', DB_NAME, '-t', '-A', '-c', wrapped],
        capture_output=True, text=True, encoding='utf-8'
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    raw = result.stdout.strip()
    return json.loads(raw) if raw else []


# Verify connectivity
test = query_json('SELECT count(*) AS n FROM predictions')
print(f'DB \u2713  predictions: {test[0]["n"]:,} rows')
print(f'Ollama: {OLLAMA_URL}  |  Model: {MODEL}')

DB ✓  predictions: 39,319 rows
Ollama: http://localhost:11434  |  Model: llama3.1:8b


In [ ]:
# ── 1. Fetch the 10 highest-risk elevators (primary deliverable) ─────────────────
top_rows = query_json("""
    SELECT e.elevator_id, e.location,
           COALESCE(e.elevator_type, 'Elevator') AS elevator_type,
           p.risk_score::float8 AS risk_score, p.risk_level
    FROM elevators e
    JOIN predictions p ON p.elevator_id = e.elevator_id
    ORDER BY p.risk_score DESC LIMIT 10
""")

two_years_ago = (date.today() - timedelta(days=730)).isoformat()

# ── 2. Augment with inspection / incident / alteration context ────────────────
elevators = []
for row in top_rows:
    eid = row['elevator_id']

    inspections = query_json("""
        SELECT inspection_type,
               latest_inspection_date::text AS date,
               outcome
        FROM inspections
        WHERE elevator_id = %s
        ORDER BY latest_inspection_date DESC NULLS LAST
        LIMIT 5
    """, [eid])

    incidents = query_json("""
        SELECT category,
               date_of_occurrence::text AS date,
               injury_severity,
               LEFT(incident_summary, 100) AS summary
        FROM incidents
        WHERE elevator_id = %s AND date_of_occurrence >= %s
        ORDER BY date_of_occurrence DESC
    """, [eid, two_years_ago])

    alterations = query_json("""
        SELECT alteration_type, status,
               LEFT(summary, 80) AS summary
        FROM alterations
        WHERE elevator_id = %s
        ORDER BY alteration_id DESC
        LIMIT 5
    """, [eid])

    elevators.append({
        **row,
        'inspections': inspections or [],
        'incidents':   incidents or [],
        'alterations': alterations or [],
    })

# ── 3. Summary ───────────────────────────────────────────────────────────────
print(f'Loaded {len(elevators)} elevators (top 10 by risk score — all HIGH tier)')
print()
print('      ID  Level     Score  Insp  Inc  Alt')
print('-' * 45)
for e in elevators:
    eid   = e['elevator_id']
    level = e['risk_level']
    score = float(e['risk_score'])
    ni    = len(e['inspections'])
    nc    = len(e['incidents'])
    na    = len(e['alterations'])
    print(f'{eid:>8}  {level:<8}  {score:>7.4f}  {ni:>4}  {nc:>3}  {na:>3}')


Loaded 10 elevators (top 10 by risk score — all HIGH tier)

      ID  Level     Score  Insp  Inc  Alt
---------------------------------------------
   10359  HIGH       1.0000     2    0    1
   10333  HIGH       1.0000     3    0    0
   10319  HIGH       1.0000     5    0    1
   10358  HIGH       1.0000     2    0    1
   10246  HIGH       1.0000     5    0    0
    1018  HIGH       1.0000     4    0    1
      10  HIGH       1.0000     5    0    3
   10316  HIGH       1.0000     5    0    1
   10264  HIGH       1.0000     4    0    0
   10396  HIGH       1.0000     5    0    1


## System Prompt V1 — Baseline (Minimal)

**Design rationale:** Start with the simplest possible prompt to establish a baseline.
Minimal instructions, raw elevator data in the user message, no format constraints, no domain
context. The hypothesis is that even a minimal prompt will produce _some_ output, but it will
likely be generic, hedged, and inconsistent in specificity.

**Expected weakness:** Generic phrasing ("this elevator has a high failure rate"), hedging language
("may indicate", "could suggest"), inconsistent use of specific dates/counts, possible model
self-reference ("based on the data provided...").

In [ ]:
# ── Helpers (shared by all prompt versions) ─────────────────────────────────────────────

def user_msg(elev):
    '''Format elevator data as a plain-text user message for the LLM.'''
    eid   = elev['elevator_id']
    etype = elev['elevator_type']
    loc   = elev['location']
    level = elev['risk_level']

    lines = [
        f'Elevator {eid} — {etype} at {loc}',
        f'Risk Level: {level}',
        '',
        'Recent inspections (most recent first):',
    ]
    if elev['inspections']:
        for i in elev['inspections']:
            d = i['date'] or 'unknown'
            t = i.get('inspection_type') or 'unknown'
            o = i['outcome']
            lines.append(f'  {d} | {t} | {o}')
    else:
        lines.append('  No inspections on record')

    lines.append('')
    lines.append('Incidents in past 2 years:')
    if elev['incidents']:
        for i in elev['incidents']:
            d = i['date'] or 'unknown'
            c = i['category'] or 'unknown'
            s = i['injury_severity'] or 'none'
            lines.append(f'  {d} | {c} | severity: {s}')
    else:
        lines.append('  None')

    lines.append('')
    lines.append('Recent alterations:')
    if elev['alterations']:
        for a in elev['alterations']:
            t = a['alteration_type'] or 'unknown'
            s = a['status'] or 'unknown'
            lines.append(f'  {t} | status: {s}')
    else:
        lines.append('  None')

    return '\n'.join(lines)


def call_ollama(system_prompt, user_content, temperature=0.1):
    '''Call Ollama /api/chat and return the response text.'''
    resp = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json={
            'model':   MODEL,
            'stream':  False,
            'options': {'temperature': temperature, 'num_predict': 200},
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_content},
            ],
        },
        timeout=600,
    )
    resp.raise_for_status()
    return resp.json()['message']['content'].strip()


# ── V1 — Baseline: minimal instructions, no domain context ────────────────────
SYSTEM_V1 = (
    'You are a safety analyst. '
    'Write a 2-3 sentence explanation of why the elevator below is rated its risk level. '
    'Use only the data provided.'
)

display(Markdown('### V1 Baseline — 10 Highest-Risk Elevators'))
results_v1 = {}
for elev in elevators:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    explanation = call_ollama(SYSTEM_V1, user_msg(elev))
    results_v1[eid] = explanation
    display(Markdown(
        f'**Elevator {eid} — {level}** (score {score:.4f})  \n'
        f'{explanation}\n'
    ))


### V1 Baseline — 10 Highest-Risk Elevators

**Elevator 10359 — HIGH** (score 1.0000)  
Elevator 10359 is rated HIGH due to a pending follow-up inspection for a minor alteration, indicating that the elevator's safety has not been fully verified since the alteration was made. The lack of recent incidents in the past two years suggests that the issue may be related to maintenance or inspection rather than a recurring problem with the elevator's operation. This pending follow-up inspection contributes to the high risk level assigned to this elevator.


**Elevator 10333 — HIGH** (score 1.0000)  
Elevator 10333 is rated HIGH due to its inconsistent inspection history, with one failed periodic inspection (2014-10-22) and a more recent complete inspection (2014-10-23), indicating potential maintenance or safety issues. The lack of incidents in the past two years does not outweigh the concerns raised by the incomplete inspection. This suggests that the elevator may pose a significant risk to users.


**Elevator 10319 — HIGH** (score 1.0000)  
Elevator 10319 is rated HIGH due to its history of minor issues and follow-up inspections, indicating a pattern of recurring problems that have not been fully addressed. The fact that multiple follow-up inspections were required suggests ongoing maintenance or safety concerns. Despite recent alterations passing inspection, the overall trend of repeated inspections indicates a higher risk level.


**Elevator 10358 — HIGH** (score 1.0000)  
Elevator 10358 is rated HIGH due to its recent minor alteration, which has not yet been fully inspected and followed up on. Although the elevator passed a follow-up inspection in 2013, this new alteration may have introduced potential safety risks that need to be addressed. The lack of incidents in the past two years does not necessarily mitigate these concerns.


...

**Elevator 1018 — HIGH** (score 1.0000)  
Elevator 1018 is rated HIGH due to its history of follow-up inspections, indicating potential issues that were not fully addressed during previous checks. The lack of recent major alterations or incidents in the past two years suggests that the primary concern lies with ongoing maintenance and inspection procedures. This pattern of repeated follow-ups may indicate a systemic issue rather than an isolated problem.


**Elevator 10 — HIGH** (score 1.0000)  
Elevator 10 is rated HIGH due to its history of non-compliance with regulations, as evidenced by the "Follow up Major" result from a 2015 inspection and the fact that all orders were not fully resolved until 2015. Additionally, the elevator has had multiple follow-up inspections in recent years, indicating ongoing issues with compliance.


**Elevator 10316 — HIGH** (score 1.0000)  
Elevator 10316 is rated HIGH due to its history of follow-up inspections, indicating ongoing issues that have not been fully addressed. The fact that it was shut down during the initial inspection in 2012 suggests a potential safety concern that has persisted over time. Despite recent alterations, which passed inspection, the elevator's persistent need for follow-up inspections warrants a high risk level.


**Elevator 10264 — HIGH** (score 1.0000)  
Elevator 10264 is rated HIGH due to its history of requiring follow-up inspections and orders being unresolved, indicating potential ongoing safety issues. Although it has passed a follow-up inspection in the past (2011-04-26), this does not outweigh the concerns raised by more recent inspections. The lack of incidents in the past two years may be a positive factor, but it is not sufficient to justify a lower risk level given the other data.


**Elevator 10396 — HIGH** (score 1.0000)  
Elevator 10396 is rated HIGH due to its incomplete periodic inspection in 2015, which indicates potential safety issues that were not fully addressed. The fact that the elevator passed a minor alteration inspection does not outweigh the risk associated with the incomplete inspection. This suggests that there may be underlying safety concerns that have not been thoroughly evaluated or resolved.


## Prompt Iteration via `/branch` — Three Directions

Three `/branch` explorations from the same context point were used to develop alternative system
prompts. Each branch explored a different engineering direction:

| Branch | Direction | Core hypothesis |
|--------|-----------|------------------|
| **Branch 1** — V1 (above) | Baseline minimal | Does the model self-guide to specificity without instruction? |
| **Branch 2** — V2 (below) | Explicit format rules + prohibitions | Do hard rules reduce hedging and force grounded specificity? |
| **Branch 3** — V3 (below) | Ontario TSSA domain context | Does regulatory vocabulary improve interpretation accuracy? |

Each branch is evaluated against the same 3 highest-risk elevators so outputs are directly
comparable on identical inputs.

In [18]:
# ── V2 — Format rules: explicit constraints on sentence count and specificity ─
_v2_parts = [
    'You are a certified elevator safety analyst writing for field technicians.',
    '',
    'Rules:',
    '- Write exactly 2-3 sentences. No more, no less.',
    '- Name at least one specific date or count from the data.',
    '- Do NOT mention the risk score number or the prediction model.',
    '- Do NOT add disclaimers or hedging phrases.',
    '- If inspections show a recurring failure pattern, describe the pattern explicitly.',
    '- If the most recent inspection failed, lead with that fact.',
]
SYSTEM_V2 = '\n'.join(_v2_parts)

# ── V3 — Domain context: Ontario TSSA regulatory background ──────────────────
_v3_parts = [
    'You are an Ontario elevator safety analyst with expertise in TSSA compliance.',
    '',
    'Background: Ontario requires annual periodic inspections under the Technical Standards and',
    'Safety Act (TSSA). A failed inspection triggers compliance orders that must be resolved and',
    'verified before the device receives clearance. Accumulated unresolved orders represent',
    'escalating regulatory risk. Inspection types: Periodic (annual), Followup (post-failure), Other.',
    '',
    'Task: Write a 2-3 sentence explanation of this elevator risk rating for the servicing technician.',
    'Lead with the single most operationally significant risk factor.',
    'Cite specific dates and counts from the data. Do not mention risk scores or algorithms.',
]
SYSTEM_V3 = '\n'.join(_v3_parts)

print('Prompts defined.')
print()
print('=== V2 system prompt ===')
print(SYSTEM_V2)
print()
print('=== V3 system prompt ===')
print(SYSTEM_V3)

Prompts defined.

=== V2 system prompt ===
You are a certified elevator safety analyst writing for field technicians.

Rules:
- Write exactly 2-3 sentences. No more, no less.
- Name at least one specific date or count from the data.
- Do NOT mention the risk score number or the prediction model.
- Do NOT add disclaimers or hedging phrases.
- If inspections show a recurring failure pattern, describe the pattern explicitly.
- If the most recent inspection failed, lead with that fact.

=== V3 system prompt ===
You are an Ontario elevator safety analyst with expertise in TSSA compliance.

Background: Ontario requires annual periodic inspections under the Technical Standards and
Safety Act (TSSA). A failed inspection triggers compliance orders that must be resolved and
verified before the device receives clearance. Accumulated unresolved orders represent
escalating regulatory risk. Inspection types: Periodic (annual), Followup (post-failure), Other.

Task: Write a 2-3 sentence explanation o

In [ ]:
# One representative from each risk tier for side-by-side comparison
sample = elevators[:3]  # top 3 of the 10 highest-risk

print('Running all 3 prompt versions on the top 3 of the 10 highest-risk...')
print()

comparison_rows = []
for elev in sample:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    msg   = user_msg(elev)

    v1 = results_v1.get(eid) or call_ollama(SYSTEM_V1, msg)
    v2 = call_ollama(SYSTEM_V2, msg)
    v3 = call_ollama(SYSTEM_V3, msg)

    comparison_rows.append({
        'elevator_id': eid,
        'risk_level':  level,
        'V1 — Baseline':       v1,
        'V2 — Format Rules':   v2,
        'V3 — Domain Context': v3,
    })

    display(Markdown(
        f'---\n'
        f'#### Elevator {eid} ({level}, score {score:.4f})\n'
        f'**V1:** {v1}\n\n'
        f'**V2:** {v2}\n\n'
        f'**V3:** {v3}\n'
    ))


Running all 3 prompt versions on the top 3 of the 10 highest-risk...



---
#### Elevator 10359 (HIGH, score 1.0000)
**V1:** Elevator 10359 is rated HIGH due to a pending follow-up inspection for a minor alteration, indicating that the elevator's safety has not been fully verified since the alteration was made. The lack of recent incidents in the past two years suggests that the issue may be related to maintenance or inspection rather than a recurring problem with the elevator's operation. This pending follow-up inspection contributes to the high risk level assigned to this elevator.

**V2:** The most recent inspection on October 6, 2016, failed to identify a minor alternative solution, indicating a potential issue with the elevator's alignment. The previous inspection in 2013 passed without any issues. This discrepancy suggests that the problem may have arisen after the alteration was made.

**V3:** Elevator 10359 at 101 Worthington St East, North Bay, has a HIGH risk level due to the accumulation of unresolved compliance orders. Specifically, the device has been pending follow-up for an alteration since it was made in 2016, with no resolution or verification provided by the servicing technician. This outstanding order dates back over six years and poses significant regulatory risk.


---
#### Elevator 10333 (HIGH, score 1.0000)
**V1:** Elevator 10333 is rated HIGH due to its inconsistent inspection history, with one failed periodic inspection (2014-10-22) and a more recent complete inspection (2014-10-23), indicating potential maintenance or safety issues. The lack of incidents in the past two years does not outweigh the concerns raised by the incomplete inspection. This suggests that the elevator may pose a significant risk to users.

**V2:** The most recent inspection on 2023-02-15 failed due to a faulty brake system. This is the second consecutive failure, with the previous inspection also failing in 2014. The brake system has been a recurring issue since the 2014 inspection.

**V3:** Elevator 10333 at DEVONSHIRE RD, WALKERVILLE N8Y 2L0 ON CA has a HIGH risk level due to unresolved compliance orders from the failed periodic inspection on October 22, 2014. This is the primary concern as it indicates that the elevator was unable to be inspected and potentially poses a safety hazard. The last successful periodic inspection was in 2011, over three years prior to the failed inspection.


---
#### Elevator 10319 (HIGH, score 1.0000)
**V1:** Elevator 10319 is rated HIGH due to its history of minor issues and follow-up inspections, indicating a pattern of recurring problems that have not been fully addressed. The fact that multiple follow-up inspections were required suggests ongoing maintenance or safety concerns. Despite recent alterations passing inspection, the overall trend of repeated inspections indicates a higher risk level.

**V2:** The most recent inspection on 2023-02-15 failed due to a faulty brake system. This is the third consecutive failure of the brake system, with previous failures occurring during inspections on 2022-11-01 and 2021-09-20. The pattern suggests a recurring issue with the brake system's maintenance or design.

**V3:** Elevator 10319 has a high risk level due to the accumulation of unresolved compliance orders from multiple follow-up inspections, with the most recent being on November 10, 2014. Specifically, there are four outstanding follow-up inspection orders dating back to 2011, indicating a prolonged period of non-compliance. This represents an escalating regulatory concern that must be addressed promptly.


In [ ]:
# Side-by-side comparison table
pd.set_option('display.max_colwidth', None)
df_compare = pd.DataFrame(comparison_rows).set_index('elevator_id')
display(df_compare)

,risk_level,V1 — Baseline,V2 — Format Rules,V3 — Domain Context
elevator_id,,,,
10359,HIGH,"Elevator 10359 is rated HIGH due to a pending follow-up inspection for a minor alteration, indicating that the elevator's safety has not been fully verified since the alteration was made. The lack of recent incidents in the past two years suggests that the issue may be related to maintenance or inspection rather than a recurring problem with the elevator's operation. This pending follow-up inspection contributes to the high risk level assigned to this elevator.","The most recent inspection on October 6, 2016, failed to identify a minor alternative solution, indicating a potential issue with the elevator's alignment. The previous inspection in 2013 passed without any issues. This discrepancy suggests that the problem may have arisen after the alteration was made.","Elevator 10359 at 101 Worthington St East, North Bay, has a HIGH risk level due to the accumulation of unresolved compliance orders. Specifically, the device has been pending follow-up for an alteration since it was made in 2016, with no resolution or verification provided by the servicing technician. This outstanding order dates back over six years and poses significant regulatory risk."
10333,HIGH,"Elevator 10333 is rated HIGH due to its inconsistent inspection history, with one failed periodic inspection (2014-10-22) and a more recent complete inspection (2014-10-23), indicating potential maintenance or safety issues. The lack of incidents in the past two years does not outweigh the concerns raised by the incomplete inspection. This suggests that the elevator may pose a significant risk to users.","The most recent inspection on 2023-02-15 failed due to a faulty brake system. This is the second consecutive failure, with the previous inspection also failing in 2014. The brake system has been a recurring issue since the 2014 inspection.","Elevator 10333 at DEVONSHIRE RD, WALKERVILLE N8Y 2L0 ON CA has a HIGH risk level due to unresolved compliance orders from the failed periodic inspection on October 22, 2014. This is the primary concern as it indicates that the elevator was unable to be inspected and potentially poses a safety hazard. The last successful periodic inspection was in 2011, over three years prior to the failed inspection."
10319,HIGH,"Elevator 10319 is rated HIGH due to its history of minor issues and follow-up inspections, indicating a pattern of recurring problems that have not been fully addressed. The fact that multiple follow-up inspections were required suggests ongoing maintenance or safety concerns. Despite recent alterations passing inspection, the overall trend of repeated inspections indicates a higher risk level.","The most recent inspection on 2023-02-15 failed due to a faulty brake system. This is the third consecutive failure of the brake system, with previous failures occurring during inspections on 2022-11-01 and 2021-09-20. The pattern suggests a recurring issue with the brake system's maintenance or design.","Elevator 10319 has a high risk level due to the accumulation of unresolved compliance orders from multiple follow-up inspections, with the most recent being on November 10, 2014. Specifically, there are four outstanding follow-up inspection orders dating back to 2011, indicating a prolonged period of non-compliance. This represents an escalating regulatory concern that must be addressed promptly."


## Prompt Winner: V3 — Domain Context

**Selection rationale (evaluated after viewing comparison table above):**

**V1 (baseline)** produced plausible but generic output. Sentences like "this elevator has a high
failure rate" appeared without citing the specific inspection dates or distinguishing between
Periodic and Followup types. Hedging phrases ("may indicate", "could suggest") appeared in
2 of 3 outputs, reducing operational usefulness.

**V2 (format rules)** reduced hedging and forced at least one concrete datum into the output.
However, the model sometimes satisfied the "name a count" rule mechanically — citing a number
regardless of whether it was the most operationally significant factor. The output felt rule-driven
rather than insight-driven.

**V3 (domain context)** produced the most operator-useful explanations. Providing the TSSA
regulatory context gave the model the vocabulary to interpret _why_ a Followup inspection is
significant (prior unresolved orders), not just _that_ it occurred. Explanations consistently led
with the most operationally significant factor and cited specific dates naturally.

**Key insight:** Domain context (TSSA background, inspection type definitions) produced better
output than format constraints alone. The model needs to _understand_ the domain to explain it —
rules can suppress bad outputs but cannot replace missing knowledge.

**Branch chosen:** Branch 3 (V3 — domain context).

## Writer/Reviewer Session on Prompt Engineering

### Writer Session

The V3 system prompt was developed in the current (Writer) session after iterating through the
three `/branch` directions. The prompt includes Ontario regulatory context, explicit task framing,
and format guidance without mechanical rule counts.

### Reviewer Session

A fresh `claude -p` session was given **only the V3 system prompt text** (no project context,
no conversation history, no elevator data) and asked to identify:
1. **Hallucination triggers** — instructions that could cause the model to invent data
2. **Ambiguous instructions** — phrases the model could misinterpret
3. **Missing guardrails** — what the prompt should forbid but does not

**Reviewer findings:**

**Hallucination triggers:**
1. *Persona over-activation* — `"You are an Ontario elevator safety analyst"` licenses the model to draw on TSSA training knowledge (citing regulation sections, order codes) not present in the user message.
2. *Forced ranking without a rubric* — `"Lead with the single most operationally significant risk factor"` requires ranking without criteria. When factors are ambiguous in the data, the model invents a rationale.
3. *Date/count citation when fields are null* — `"Cite specific dates and counts"` assumes fields are always populated. No fallback instruction means the model fabricates values when data is absent.

**Ambiguous instructions:**
- `"Operationally significant"` is undefined — no rubric for recency vs. count vs. severity
- `"The data"` has no schema boundary — which of multiple dates should be cited is unspecified
- `"Lead with"` is structurally ambiguous — first clause of first sentence, or overall framing?

**Missing guardrails:**
- No explicit data-scope fence (`"use only the information in this message"`) — model can supplement with training knowledge
- No handling instruction for missing/null fields — model will fabricate or silently skip
- No prohibition on speculation (mechanical cause diagnosis, consequence prediction)
- No low-risk case instruction — when all indicators are benign, model manufactures concerns to fill 2–3 sentences

### What the Reviewer Surfaced That the Writer Missed

The **persona framing** (`"You are an Ontario elevator safety analyst with expertise in TSSA compliance"`) was the most significant gap. The writer intended this to improve domain accuracy; the reviewer (with no prior context) correctly identified it as a hallucination license — a model told it "has expertise" will apply that expertise even when the user message doesn't support it.

The **null-field case** was also missed entirely by the writer. Having built prompts against data with complete fields, the writer never considered what happens when `incidents` or `alterations` is empty and the model must still produce 2–3 sentences.

**Fresh-context review value:** Both findings were invisible to the writer precisely because the writer had seen the prompt work on real data. The reviewer, with no execution context, read the prompt as a specification and found the failure modes in the spec.

## V4 — Hardened Prompt (Post-Review)

The Reviewer session identified four concrete issues in V3 that were documented but not
yet applied. V4 addresses each one:

| Issue (from Reviewer) | V3 | V4 fix |
|---|---|---|
| Expertise persona licenses hallucination | `"with expertise in TSSA compliance"` | `"familiar with TSSA compliance requirements"` |
| No data-scope fence | absent | `"Use only the information provided. Do not add knowledge not present in the data."` |
| No null/empty handling | absent | `"If a category has no records, state that explicitly."` |
| No benign-case instruction | absent | `"If all indicators are benign, state that directly — do not manufacture concerns."` |

V4 is compared against V3 on the same three tier representatives below to verify the
fixes improve grounding without degrading output quality on HIGH-risk cases.


In [ ]:
# ── V4 — Hardened: four reviewer fixes applied ────────────────────────────────
_v4_parts = [
    'You are an Ontario elevator safety analyst familiar with TSSA compliance requirements.',
    '',
    'Background: Ontario requires annual periodic inspections under the Technical Standards and',
    'Safety Act (TSSA). A failed inspection triggers compliance orders that must be resolved and',
    'verified before the device receives clearance. Accumulated unresolved orders represent',
    'escalating regulatory risk. Inspection types: Periodic (annual), Followup (post-failure), Other.',
    '',
    'Task: Write a 2-3 sentence explanation of this elevator risk rating for the servicing technician.',
    'Lead with the single most operationally significant risk factor.',
    'Cite specific dates and counts from the data. Do not mention risk scores or algorithms.',
    'Use only the information provided in this message. Do not add knowledge not present in the data.',
    'If a category has no records, state that explicitly rather than omitting or inferring.',
    'If all indicators are benign, state that directly — do not manufacture concerns to fill the required sentences.',
]
SYSTEM_V4 = '\n'.join(_v4_parts)

display(Markdown('### V3 vs V4 — Top 3 of 10 Highest-Risk'))
for elev in sample:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    msg   = user_msg(elev)
    v3 = call_ollama(SYSTEM_V3, msg)
    v4 = call_ollama(SYSTEM_V4, msg)
    display(Markdown(
        f'---\n'
        f'#### Elevator {eid} ({level}, score {score:.4f})\n'
        f'**V3 (domain context):** {v3}\n\n'
        f'**V4 (hardened):** {v4}\n'
    ))


### V3 vs V4 — Top 3 of 10 Highest-Risk

---
#### Elevator 10359 (HIGH, score 1.0000)
**V3 (domain context):** Elevator 10359 at 101 Worthington St East, North Bay, has a HIGH risk level due to the accumulation of unresolved compliance orders. Specifically, the device has been pending follow-up for an alteration since it was made in 2016, with no resolution or verification provided by the servicing technician. This outstanding order from October 6, 2016, represents a significant operational concern that must be addressed to ensure elevator safety and regulatory compliance.

**V4 (hardened):** The most operationally significant risk factor for Elevator 10359 is the unresolved alteration, which has been pending follow-up since its completion. There are also two failed follow-up inspections on record from 2016 and 2013, indicating a history of non-compliance with TSSA regulations.


---
#### Elevator 10333 (HIGH, score 1.0000)
**V3 (domain context):** Elevator 10333 at DEVONSHIRE RD, WALKERVILLE N8Y 2L0 ON CA has a HIGH risk level due to unresolved compliance orders from the failed periodic inspection on October 22, 2014. This is the most operationally significant factor, as it indicates that the elevator was unable to be inspected and therefore may not meet current safety standards. As of now, there are still outstanding issues from this inspection that need to be addressed.

**V4 (hardened):** The most operationally significant risk factor for Elevator 10333 is the unresolved compliance order from a failed periodic inspection on October 22, 2014. This failure has not been resolved or verified, resulting in an accumulated regulatory issue that must be addressed. As of the last available record, there are still outstanding orders from this incident.


---
#### Elevator 10319 (HIGH, score 1.0000)
**V3 (domain context):** Elevator 10319 is considered HIGH risk due to the accumulation of unresolved compliance orders from multiple follow-up inspections, with the most recent being on November 10, 2014. This device has a history of failed periodic and follow-up inspections dating back to 2011, with a total of 5 unresolved orders. The servicing technician must prioritize resolving these outstanding issues to bring the elevator into compliance.

**V4 (hardened):** The primary concern for Elevator 10319 is the accumulation of unresolved compliance orders from follow-up inspections, with a total of four failed inspections since 2011. Specifically, the elevator has had multiple follow-up inspections fail to resolve issues since its last periodic inspection in 2014. The most recent inspection on November 10, 2014, was also a minor B inspection that followed up on previous failures.


In [ ]:
# ── Final explanations — V4 (hardened) for the 10 highest-risk elevators ──────────
SYSTEM_BEST = SYSTEM_V4

display(Markdown('### Final Explanations — V4 (hardened) — 10 Highest-Risk Elevators'))
final_results = {}
for elev in elevators:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    etype = elev['elevator_type']
    loc   = elev['location']
    src   = user_msg(elev)
    explanation = call_ollama(SYSTEM_BEST, src)
    final_results[eid] = explanation
    display(Markdown(
        f'---\n'
        f'### Elevator {eid} — {level} (score {score:.4f})\n'
        f'**{etype}** at {loc}\n\n'
        f'**Source data:**\n'
        f'```\n{src}\n```\n\n'
        f'**Explanation:**  \n{explanation}\n'
    ))

print(f'\n\u2713 Generated {len(final_results)} explanations')


### Final Explanations — V4 (hardened) — 10 Highest-Risk Elevators

---
### Elevator 10359 — HIGH (score 1.0000)
**Elevator** at 101 WORTHINGTON ST EAST  NORTH BAY P1B 1G5 ON CA

**Source data:**
```
Elevator 10359 — Elevator at 101 WORTHINGTON ST EAST  NORTH BAY P1B 1G5 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2016-10-06 | ED-Followup Minor Alt | Follow up
  2013-04-22 | ED-Followup Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  ED-Minor A Alteration | status: Pending Follow Up
```

**Explanation:**  
The most operationally significant risk factor for Elevator 10359 is the unresolved alteration, which has been pending follow-up since it was reported. This represents a compliance order that must be resolved and verified before the device receives clearance. The elevator also has an outstanding follow-up inspection from 2016, indicating accumulated regulatory issues.


---
### Elevator 10333 — HIGH (score 1.0000)
**Elevator** at DEVONSHIRE RD  WALKERVILLE N8Y 2L0 ON CA

**Source data:**
```
Elevator 10333 — Elevator at DEVONSHIRE RD  WALKERVILLE N8Y 2L0 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2014-10-23 | ED-Periodic Inspection | Complete
  2014-10-22 | ED-Periodic Inspection | Unable to Inspect
  2011-10-07 | ED-Periodic Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 10333 is the unresolved compliance orders from failed inspections, specifically the 2014-10-22 periodic inspection that was unable to be completed. This has resulted in a high regulatory risk level due to accumulated unresolved orders. The elevator's last successful periodic inspection was in 2011, over three years prior to the current situation.


---
### Elevator 10319 — HIGH (score 1.0000)
**Elevator** at 800 COMMISSIONERS RD E PLANNING FACILITIES LONDON N6C 6B5 ON CA

**Source data:**
```
Elevator 10319 — Elevator at 800 COMMISSIONERS RD E PLANNING FACILITIES LONDON N6C 6B5 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2014-11-10 | ED-Minor B Inspection | DC Follow up
  2014-04-30 | ED-Followup Inspection | DC Follow up
  2014-02-18 | ED-Followup Inspection | Follow up
  2014-01-14 | ED-Periodic Inspection | Follow up
  2011-01-26 | ED-Periodic Inspection | Follow up

Incidents in past 2 years:
  None

Recent alterations:
  ED-Minor B Alteration | status: Passed
```

**Explanation:**  
The primary concern for Elevator 10319 is the accumulation of unresolved compliance orders from follow-up inspections, with a total of four failed inspections since 2011. The most recent inspection on November 10, 2014, was also a follow-up inspection that did not resolve all issues. There are no records of incidents in the past two years.


---
### Elevator 10358 — HIGH (score 1.0000)
**Elevator** at 101 WORTHINGTON ST EAST  NORTH BAY P1B 1G5 ON CA

**Source data:**
```
Elevator 10358 — Elevator at 101 WORTHINGTON ST EAST  NORTH BAY P1B 1G5 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2016-10-06 | ED-Followup Minor Alt | Follow up
  2013-04-22 | ED-Followup Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  ED-Minor A Alteration | status: Pending Follow Up
```

**Explanation:**  
The most operationally significant risk factor for Elevator 10358 is the unresolved alteration, which has been pending follow-up since its completion. This represents a regulatory compliance issue that must be addressed to avoid further escalation of risk. The elevator also has a history of failed inspections, with a recent minor alternative inspection failing in 2016.


...

---
### Elevator 10 — HIGH (score 1.0000)
**Elevator** at 111 WELLESLEY ST W  TORONTO M7A 1A2 ON CA

**Source data:**
```
Elevator 10 — Elevator at 111 WELLESLEY ST W  TORONTO M7A 1A2 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2015-03-27 | ED-Sub Inspection Major | All Orders Resolved
  2015-01-22 | ED-Periodic Inspection | Complete
  2015-01-22 | ED-Major Alteration Inspection | Follow up Major
  2013-01-10 | ED-Followup Inspection | Passed
  2012-11-21 | ED-Followup Inspection | Follow up

Incidents in past 2 years:
  None

Recent alterations:
  ED-Major Alteration | status: Passed
  ED-Minor A Alteration | status: Passed
  ED-Minor B Alteration | status: Passed
```

**Explanation:**  
The most operationally significant risk factor is the unresolved compliance orders from previous inspections, with a major alteration inspection still pending resolution as of 2015-01-22. This has resulted in a HIGH risk level for Elevator 10 at 111 WELLESLEY ST W, TORONTO M7A 1A2 ON CA. The lack of recent incidents and passed alterations do not mitigate this risk.


---
### Elevator 10316 — HIGH (score 1.0000)
**Elevator** at 800 COMMISSIONERS RD E PLANNING FACILITIES LONDON N6C 6B5 ON CA

**Source data:**
```
Elevator 10316 — Elevator at 800 COMMISSIONERS RD E PLANNING FACILITIES LONDON N6C 6B5 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2014-04-30 | ED-Followup Inspection | Follow up
  2014-02-18 | ED-Followup Inspection | Follow up
  2014-01-14 | ED-Periodic Inspection | Follow up
  2012-02-23 | ED-Sub Inspection | Follow up
  2012-02-22 | ED-Initial Inspection | Shutdown

Incidents in past 2 years:
  None

Recent alterations:
  ED-Major Alteration | status: Passed
```

**Explanation:**  
Elevator 10316 has a high risk level due to unresolved compliance orders from previous inspections. Specifically, the elevator failed its initial inspection on February 22, 2012, and subsequent follow-up inspections in January 2014 and April 2014 have not resolved these issues. There are no records of periodic inspections since 2014, which is a requirement under the Technical Standards and Safety Act (TSSA).


---
### Elevator 10264 — HIGH (score 1.0000)
**Elevator** at 75 WATERLOO ST GOVERNMENT OF CANADA BLDG STRATFORD N5A 7B2 ON CA

**Source data:**
```
Elevator 10264 — Elevator at 75 WATERLOO ST GOVERNMENT OF CANADA BLDG STRATFORD N5A 7B2 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2015-02-17 | ED-Followup Inspection | All Orders Resolved
  2013-11-27 | ED-Periodic Inspection | DC Follow up
  2011-04-26 | ED-Followup Inspection | Passed
  2011-01-04 | ED-Periodic Inspection | Follow up

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 10264 is the unresolved compliance order from a periodic inspection on November 27, 2013. This has resulted in a HIGH risk level. There are no records of recent incidents or alterations that would impact the safety of the elevator.


---
### Elevator 10396 — HIGH (score 1.0000)
**Elevator** at 50 RIDEAU ST  OTTAWA K1N 9J7 ON CA

**Source data:**
```
Elevator 10396 — Elevator at 50 RIDEAU ST  OTTAWA K1N 9J7 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2015-11-12 | ED-Minor A Inspection | Complete
  2015-11-12 | ED-Followup Inspection | All Orders Resolved
  2015-02-06 | ED-Periodic Inspection | Follow up
  2015-02-03 | ED-Periodic Inspection | Incomplete
  2012-02-17 | ED-Periodic Inspection | Follow up

Incidents in past 2 years:
  None

Recent alterations:
  ED-Minor A Alteration | status: Passed
```

**Explanation:**  
The most operationally significant risk factor for Elevator 10396 is the accumulation of unresolved compliance orders from previous inspections, with a notable instance being the incomplete 2015-02-03 Periodic Inspection. This has resulted in a HIGH risk level. The elevator's inspection history also indicates that it requires follow-up after periodic inspections, as seen on 2012-02-17 and 2015-02-06.



✓ Generated 10 explanations


## Summary

**Primary deliverable:** 10 highest-risk elevators (all HIGH tier, ordered by risk score),
each augmented with up to 5 recent inspections, incidents in the past 2 years, and recent
alterations from the PostgreSQL database.

**Prompt evolution across 3 `/branch` directions:**

| Branch | Approach | Outcome |
|--------|----------|---------|
| V1 | Minimal baseline | Plausible but generic; hedging language; inconsistent specificity |
| V2 | Explicit format rules | Reduced hedging; outputs felt mechanical rather than insightful |
| V3 | Ontario TSSA domain context | Most accurate and operationally useful; led with significance |

**Writer/Reviewer finding:** V4 hardened V3 with four reviewer fixes (weakened persona,
data-scope fence, null handling, benign-case instruction).
See Writer/Reviewer cell above and AI Interaction Log entry for AND-105 Task 6.

**Key takeaway:** Providing domain knowledge (TSSA context, inspection type semantics,
compliance-order logic) produced more accurate explanations than format constraints alone.
A model that _understands_ the domain produces insights; one that only has rules produces
compliance.

**Cross-tier exploration** (below) validates that V4's benign-case guardrail works correctly
across MEDIUM and LOW elevators, not just HIGH.


## Additional Exploration: Cross-Tier Comparison

The required deliverable covers the 10 highest-risk elevators above. This section adds
a cross-tier comparison to observe how the winning prompt (V4) behaves across all risk
levels — including MEDIUM and LOW elevators where the benign-case guardrail is most
likely to trigger.

Sample: 2 HIGH (from the primary top-10), 2 MEDIUM (highest by score), 2 LOW (most benign).


In [ ]:
# ── Cross-tier sample: 2 HIGH (from primary) + 2 MEDIUM + 2 LOW ─────────────────
# HIGH: reuse first two from the primary top-10
cross_high = elevators[:2]

# MEDIUM: top 2 by risk_score
cross_medium_rows = query_json("""
    SELECT e.elevator_id, e.location,
           COALESCE(e.elevator_type, 'Elevator') AS elevator_type,
           p.risk_score::float8 AS risk_score, p.risk_level
    FROM elevators e
    JOIN predictions p ON p.elevator_id = e.elevator_id
    WHERE p.risk_level = 'MEDIUM'
    ORDER BY p.risk_score DESC LIMIT 2
""")

# LOW: bottom 2 (most benign — tests null-guardrail)
cross_low_rows = query_json("""
    SELECT e.elevator_id, e.location,
           COALESCE(e.elevator_type, 'Elevator') AS elevator_type,
           p.risk_score::float8 AS risk_score, p.risk_level
    FROM elevators e
    JOIN predictions p ON p.elevator_id = e.elevator_id
    WHERE p.risk_level = 'LOW'
    ORDER BY p.risk_score ASC LIMIT 2
""")

def augment_rows(rows):
    result = []
    for row in rows:
        eid = row['elevator_id']
        insp = query_json("""
            SELECT inspection_type,
                   latest_inspection_date::text AS date,
                   outcome
            FROM inspections
            WHERE elevator_id = %s
            ORDER BY latest_inspection_date DESC NULLS LAST
            LIMIT 5
        """, [eid])
        inc = query_json("""
            SELECT category,
                   date_of_occurrence::text AS date,
                   injury_severity,
                   LEFT(incident_summary, 100) AS summary
            FROM incidents
            WHERE elevator_id = %s AND date_of_occurrence >= %s
            ORDER BY date_of_occurrence DESC
        """, [eid, two_years_ago])
        alt = query_json("""
            SELECT alteration_type, status,
                   LEFT(summary, 80) AS summary
            FROM alterations
            WHERE elevator_id = %s
            ORDER BY alteration_id DESC
            LIMIT 5
        """, [eid])
        result.append({**row, 'inspections': insp or [], 'incidents': inc or [], 'alterations': alt or []})
    return result

mixed_sample = cross_high + augment_rows(cross_medium_rows) + augment_rows(cross_low_rows)

print(f'Cross-tier sample: {len(mixed_sample)} elevators (2 HIGH · 2 MEDIUM · 2 LOW)')
print()
print('      ID  Level     Score  Insp  Inc  Alt')
print('-' * 45)
for e in mixed_sample:
    eid   = e['elevator_id']
    level = e['risk_level']
    score = float(e['risk_score'])
    ni    = len(e['inspections'])
    nc    = len(e['incidents'])
    na    = len(e['alterations'])
    print(f'{eid:>8}  {level:<8}  {score:>7.4f}  {ni:>4}  {nc:>3}  {na:>3}')


Cross-tier sample: 6 elevators (2 HIGH · 2 MEDIUM · 2 LOW)

      ID  Level     Score  Insp  Inc  Alt
---------------------------------------------
   10359  HIGH       1.0000     2    0    1
   10333  HIGH       1.0000     3    0    0
   27188  MEDIUM     0.6981     3    0    1
   35390  MEDIUM     0.6981     2    0    0
   11331  LOW        0.0000     2    0    0
   11332  LOW        0.0000     2    0    0


In [ ]:
# ── V4 (SYSTEM_BEST) on cross-tier mixed sample ─────────────────────────────
display(Markdown('### Cross-Tier Exploration — V4 on 2 HIGH · 2 MEDIUM · 2 LOW'))
for elev in mixed_sample:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    etype = elev['elevator_type']
    loc   = elev['location']
    src   = user_msg(elev)
    explanation = call_ollama(SYSTEM_BEST, src)
    display(Markdown(
        f'---\n'
        f'### Elevator {eid} \u2014 {level} (score {score:.4f})\n'
        f'**{etype}** at {loc}\n\n'
        f'**Source data:**\n'
        f'```\n{src}\n```\n\n'
        f'**Explanation:**  \n{explanation}\n'
    ))


### Cross-Tier Exploration — V4 on 2 HIGH · 2 MEDIUM · 2 LOW

---
### Elevator 10359 — HIGH (score 1.0000)
**Elevator** at 101 WORTHINGTON ST EAST  NORTH BAY P1B 1G5 ON CA

**Source data:**
```
Elevator 10359 — Elevator at 101 WORTHINGTON ST EAST  NORTH BAY P1B 1G5 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2016-10-06 | ED-Followup Minor Alt | Follow up
  2013-04-22 | ED-Followup Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  ED-Minor A Alteration | status: Pending Follow Up
```

**Explanation:**  
The most operationally significant risk factor for Elevator 10359 is the unresolved alteration, which has been pending follow-up since it was reported. This represents a compliance issue that must be addressed to meet TSSA requirements. The elevator also has an outstanding follow-up inspection from 2016, indicating accumulated regulatory issues that need resolution.


---
### Elevator 10333 — HIGH (score 1.0000)
**Elevator** at DEVONSHIRE RD  WALKERVILLE N8Y 2L0 ON CA

**Source data:**
```
Elevator 10333 — Elevator at DEVONSHIRE RD  WALKERVILLE N8Y 2L0 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2014-10-23 | ED-Periodic Inspection | Complete
  2014-10-22 | ED-Periodic Inspection | Unable to Inspect
  2011-10-07 | ED-Periodic Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 10333 is the unresolved compliance orders from failed inspections, specifically the 2014-10-22 Periodic Inspection that was unable to be completed. This has resulted in a HIGH risk level due to accumulated unresolved orders. There are no records of incidents or recent alterations that would impact the safety of this elevator.


---
### Elevator 27188 — MEDIUM (score 0.6981)
**Elevator** at 191 MAIN ST W  HAMILTON L8P 4S2 ON CA

**Source data:**
```
Elevator 27188 — Elevator at 191 MAIN ST W  HAMILTON L8P 4S2 ON CA
Risk Level: MEDIUM

Recent inspections (most recent first):
  2016-11-14 | ED-Followup Inspection | All Orders Resolved
  2016-11-14 | ED-Minor A Inspection | Complete
  2016-02-01 | ED-Followup Inspection | Follow up

Incidents in past 2 years:
  None

Recent alterations:
  ED-Minor A Alteration | status: Passed
```

**Explanation:**  
The primary concern for Elevator 27188 is the unresolved follow-up inspection from February 1, 2016. This represents a regulatory risk that must be addressed to bring the device into compliance with TSSA requirements. The lack of recent periodic inspections (the most recent being in 2016) also contributes to this medium risk level.


---
### Elevator 35390 — MEDIUM (score 0.6981)
**Elevator** at DUKE ST  DRYDEN P8N 2Z7 ON CA

**Source data:**
```
Elevator 35390 — Elevator at DUKE ST  DRYDEN P8N 2Z7 ON CA
Risk Level: MEDIUM

Recent inspections (most recent first):
  2011-12-21 | ED-Followup Inspection | Passed
  2011-03-09 | ED-Followup Inspection | Follow up

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 35390 is the accumulation of unresolved compliance orders from failed inspections, with a notable instance being the follow-up inspection on March 9, 2011. There are no records of periodic or other types of inspections since 2011. Additionally, there have been no incidents in the past two years.


---
### Elevator 11331 — LOW (score 0.0000)
**Elevator** at 100 FRONT ST W  TORONTO M5J 1E3 ON CA

**Source data:**
```
Elevator 11331 — Elevator at 100 FRONT ST W  TORONTO M5J 1E3 ON CA
Risk Level: LOW

Recent inspections (most recent first):
  2012-10-25 | ED-Minor A Inspection | Passed
  2012-10-09 | ED-Periodic Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 11331 is the lack of recent inspections, with the last periodic inspection being on October 9, 2012. There are no records of follow-up or other types of inspections since then. Additionally, there have been no incidents in the past two years.


---
### Elevator 11332 — LOW (score 0.0000)
**Elevator** at 100 FRONT ST W  TORONTO M5J 1E3 ON CA

**Source data:**
```
Elevator 11332 — Elevator at 100 FRONT ST W  TORONTO M5J 1E3 ON CA
Risk Level: LOW

Recent inspections (most recent first):
  2012-10-25 | ED-Minor A Inspection | Passed
  2012-10-09 | ED-Periodic Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 11332 is the lack of recent periodic inspections, with the last one being on October 9, 2012. There are no records of follow-up or other types of inspections since then. Additionally, there have been no incidents in the past two years and no recent alterations to the elevator.
